# 99 - Master pipeline launcher (the runner decides every next step)

**Private compute session - not a dataset mirror.** Raw BraTS data come only from the official TCIA source into ephemeral session storage (`WORK`) and are never written to notebook output, a Kaggle Dataset, GitHub or the website. B1 basis: owner-approved alternative (amendment v1.0-A1); no TCIA authorization is claimed. Enable **GPU** and **Internet** in the notebook settings. Never paste credentials into a cell; use Kaggle Secrets if a private repository needs a token.

Launches `scripts/remote/master_run.py`, the single entry point. The runner (brats_uncertainty.orchestration, tested) re-derives every gate from `docs/project_status.yaml`, runs each permitted step in protocol order, resumes interrupted work, commits and pushes milestones (allow-listed, public-safe paths only), and stops only at a genuine blocker with the exact gate, blocker and action. **The preferred full-study environment is a private persistent GPU VM** (`scripts/remote/run_all.sh`); Kaggle session storage is ephemeral, so raw data are re-acquired per session and verified against the committed B2 inventory.

In [ ]:
# PARAMETERS - set before running (no secrets here)
REPO_URL = ""   # https URL of this research repository (GitHub)
BRANCH = "main"  # the runner pushes milestones here
WORK = "/tmp/brats"                          # ephemeral raw-data storage
STATE = "/kaggle/working/master_state"       # runner journal (output)
assert REPO_URL, "set REPO_URL"

In [ ]:
import os

!git clone --quiet --branch {BRANCH} {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git rev-parse HEAD
!pip install --quiet -e ".[io,nnunet]"
!brats-uncertainty verify-protocol
os.environ["BRATS_WORK"] = WORK
os.environ["BRATS_STATE_DIR"] = STATE

In [ ]:
# Push credential from Kaggle Secrets (never printed, never written to disk)
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("no GITHUB_TOKEN secret: milestones are committed but not pushed")

In [ ]:
# Official IBM Aspera CLI (needed only for a runtime receive at B2)
!apt-get -qq update && apt-get -qq install -y ruby-full > /dev/null
!gem install aspera-cli --no-document
!ascli config transferd install
!ascli config ascp show

In [ ]:
PUSH = "--push" if os.environ.get("GITHUB_TOKEN") else ""
!python scripts/remote/master_run.py --resume --commit {PUSH}